# Building an MCP Server in Python
Companion notebook for the **Building an MCP Server in Python** section of Chapter 5.

An MCP server is a long-running process (it blocks on `mcp.run(...)`), so it cannot live inside a normal notebook cell the way the other examples do. Instead, this notebook **writes `server.py` to the same folder** using the `%%writefile` magic -- run this notebook once and the file appears on disk, exactly the way the book's GitHub repo ships it. Notebooks 04 and 05 both launch this file as a subprocess, so **run this notebook before those two**.

> One deliberate change from the book text: `search_web`'s `TavilySearchResults` client is now built *inside* the tool function instead of at module import time. `TavilySearchResults` validates `TAVILY_API_KEY` the moment it's constructed -- building it eagerly at import time would crash the whole server process on startup whenever the key isn't set, which takes `search_wikipedia` down with it. Building it lazily means only `search_web` itself fails (with a clear message) when the key is missing.

In [1]:
import os
os.makedirs(".", exist_ok=True)  # writefile below targets the notebook's own directory

In [2]:
%%writefile server.py
import os
import sys
from mcp.server.fastmcp import FastMCP
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv

load_dotenv()

mcp = FastMCP("SearchServer")

wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())


@mcp.tool()
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for information about people, places, facts, and history."""
    return wikipedia_tool.invoke(query)


@mcp.tool()
def search_web(query: str) -> str:
    """Search the web for current events, news, or general queries using Tavily."""
    # Built lazily: TavilySearchResults validates TAVILY_API_KEY at construction, which
    # would otherwise crash the whole server process on startup when the key is unset.
    if not os.environ.get("TAVILY_API_KEY"):
        return "TAVILY_API_KEY not set - cannot search the web."
    tavily_tool = TavilySearchResults(max_results=3, search_depth="advanced")
    result = tavily_tool.invoke({"query": query})
    return str(result)


if __name__ == "__main__":
    protocol = sys.argv[1] if len(sys.argv) > 1 else "stdio"
    if protocol == "sse":
        print("Starting MCP Server with SSE transport on http://localhost:8000/sse")
        mcp.run(transport="sse")
    else:
        mcp.run(transport="stdio")

Writing server.py


Both tools are registered with the `@mcp.tool()` decorator -- `FastMCP` inspects each function's signature and docstring to build the tool descriptor (name, description, input schema) automatically, the same shape shown in notebook 02.

The bottom block picks the transport from the command line: `python server.py` (defaults to `stdio`) for a local subprocess, or `python server.py sse` for a network-reachable server.

In [3]:
from pathlib import Path

assert Path("server.py").exists()
print("server.py written to:", Path("server.py").resolve())

server.py written to: C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\output5\server.py
